# Predict Notebook for Time Measurement

In [ ]:
import os
import torch
import random
import numpy as np
import time
import json
from pathlib import Path

def seed_everything(seed=42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_everything(42)

In [ ]:
# Load model and resources
from predict import YOLODetector
from config import TRAINED_MODEL_PATH, CONFIDENCE_THRESHOLD

# Assuming model path is set correctly in config or passed here
# In docker, model should be in /code/saved_models/
model_path = "/code/saved_models/models.safetensors" # Update this if your model name differs
if not os.path.exists(model_path):
    # Fallback to config path if specific file not found
    model_path = TRAINED_MODEL_PATH

detector = YOLODetector(model_path, confidence=CONFIDENCE_THRESHOLD)

In [ ]:
# Read test cases
test_root = "/data/samples"
video_dirs = sorted([d for d in Path(test_root).iterdir() if d.is_dir()])
print(f"Found {len(video_dirs)} test cases")

In [ ]:
# Predict and measure time
all_predicted_time = []
all_result = []

print("Starting inference...")

for vid_dir in video_dirs:
    video_path = vid_dir / "drone_video.mp4"
    file_name = vid_dir.name
    
    t1 = time.time()
    
    # Preprocessing (if any) is handled inside detect_video or separate call
    # Here we assume detect_video handles reading the video
    detections = detector.detect_video(str(video_path))
    
    # Postprocess result
    result = {
        "video_id": file_name,
        "detections": [{"bboxes": detections}] if detections else []
    }
    
    t2 = time.time()
    predicted_time = int((t2 - t1) * 1000)
    
    all_predicted_time.append((file_name, predicted_time))
    all_result.append(result)

# Write outputs
with open("/result/jupyter_submission.json", "w") as f:
    json.dump(all_result, f, indent=2)

import csv
with open("/result/time_submission.csv", "w", newline='') as f:
    writer = csv.writer(f)
    writer.writerow(["id", "answer", "time"])
    # Note: 'answer' column content is not clearly specified in guideline for time_submission.csv,
    # but usually it's the prediction or just a placeholder if checked against submission.json.
    # The guideline says "format gồm 3 cột là id, answer, và time".
    # We will leave 'answer' empty or put a placeholder as we are writing the full result to json.
    for file_name, p_time in all_predicted_time:
        writer.writerow([file_name, "", p_time])

print("Done.")